# Lesson 7.2: Color Channel Processing
## Biomedical Image Processing - Color Image Processing

### Topics:
- Per-channel operations
- Channel arithmetic
- Color image brightness and contrast
- Channel swapping and manipulation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Create a colorful test image (200x300x3)
img = np.zeros((200, 300, 3), dtype=np.uint8)

# Background gradient: red increases left-to-right, blue increases top-to-bottom
for i in range(200):
    for j in range(300):
        img[i, j, 0] = int(j / 300 * 255)   # Red gradient left-to-right
        img[i, j, 2] = int(i / 200 * 255)   # Blue gradient top-to-bottom
        img[i, j, 1] = 50                     # Slight green base

# Draw a red circle (center 70,80, radius 40)
Y, X = np.ogrid[:200, :300]
mask_circle1 = (X - 80)**2 + (Y - 70)**2 < 40**2
img[mask_circle1] = [255, 30, 30]

# Draw a green circle (center 70,220, radius 40)
mask_circle2 = (X - 220)**2 + (Y - 70)**2 < 40**2
img[mask_circle2] = [30, 255, 30]

# Draw a blue rectangle
img[130:180, 30:120] = [30, 30, 255]

# Draw a yellow rectangle
img[130:180, 180:270] = [255, 255, 30]

# Display the test image
plt.figure(figsize=(6, 4))
plt.imshow(img)
plt.title("Test Color Image")
plt.axis("off")
plt.tight_layout()
plt.show()

print(f"Image shape: {img.shape}, dtype: {img.dtype}")

## 1. Per-Channel Processing

A color image is a stack of 3 grayscale images (R, G, B). We can process each channel independently using any grayscale transformation $T$:

$$g_R(x,y) = T[f_R(x,y)], \quad g_G(x,y) = T[f_G(x,y)], \quad g_B(x,y) = T[f_B(x,y)]$$

This is the simplest approach to color image processing — apply the same (or different) operations to each channel separately.

In [ ]:
# Extract individual color channels
R = img[:, :, 0]  # Red channel
G = img[:, :, 1]  # Green channel
B = img[:, :, 2]  # Blue channel

# Display original and individual channels
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(img)
axes[0].set_title("Original")

axes[1].imshow(R, cmap="Reds")
axes[1].set_title("Red Channel")

axes[2].imshow(G, cmap="Greens")
axes[2].set_title("Green Channel")

axes[3].imshow(B, cmap="Blues")
axes[3].set_title("Blue Channel")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 2. Brightness Adjustment on Color Images

There are two main approaches to adjusting brightness in a color image:

1. **Add a constant to all RGB channels equally** — simple but can shift the perceived color (hue).
2. **Convert to HSI, modify Intensity only, convert back** — preserves the hue and saturation.

The HSI approach is generally preferred in biomedical imaging where color accuracy matters.

In [ ]:
# Approach 1: Add constant to all RGB channels
bright_rgb = np.clip(img.astype(np.int16) + 50, 0, 255).astype(np.uint8)

# Approach 2: Modify intensity in HSI space
# Convert to float [0, 1]
img_float = img.astype(np.float64) / 255.0

# Compute intensity channel: I = (R + G + B) / 3
intensity = np.mean(img_float, axis=2)

# Increase intensity by a factor, then scale RGB to match
# New intensity
new_intensity = np.clip(intensity + 0.2, 0, 1)

# Scale each pixel's RGB by (new_I / old_I), avoid division by zero
scale = np.where(intensity > 0.001, new_intensity / intensity, 1.0)
bright_hsi = np.clip(img_float * scale[:, :, np.newaxis], 0, 1)
bright_hsi = (bright_hsi * 255).astype(np.uint8)

# Display results
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(img)
axes[0].set_title("Original")

axes[1].imshow(bright_rgb)
axes[1].set_title("RGB + 50 (all channels)")

axes[2].imshow(bright_hsi)
axes[2].set_title("HSI Intensity + 0.2\n(preserves hue better)")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 3. Contrast Adjustment on Color Images

Per-channel contrast adjustment multiplies each channel by a scaling factor $c$:

$$g(x,y) = c \cdot f(x,y)$$

- $c < 1$: reduces contrast (image becomes darker/flatter)
- $c > 1$: increases contrast (image becomes brighter/more vivid)

Values must be clipped to $[0, 255]$ to avoid overflow.

In [ ]:
# Contrast adjustment: multiply all channels by a constant
low_contrast = np.clip(img.astype(np.float64) * 0.5, 0, 255).astype(np.uint8)
high_contrast = np.clip(img.astype(np.float64) * 1.5, 0, 255).astype(np.uint8)

# Display results
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(img)
axes[0].set_title("Original (c = 1.0)")

axes[1].imshow(low_contrast)
axes[1].set_title("Low Contrast (c = 0.5)")

axes[2].imshow(high_contrast)
axes[2].set_title("High Contrast (c = 1.5)")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 4. Channel Arithmetic

We can combine channels mathematically to extract useful information:

- **Add/subtract channels** to highlight color differences
- **Multiply channels** for masking operations
- **Channel difference** for detecting where one color dominates
- **Channel averaging** for grayscale conversion

In [ ]:
# Channel difference: highlight where one color dominates
R = img[:, :, 0].astype(np.float64)
G = img[:, :, 1].astype(np.float64)
B = img[:, :, 2].astype(np.float64)

# R - G: positive = red-dominant, negative = green-dominant
diff_rg = R - G
# R - B: positive = red-dominant, negative = blue-dominant
diff_rb = R - B

# Grayscale conversion via channel averaging
gray_avg = ((R + G + B) / 3.0).astype(np.uint8)

# Weighted grayscale (luminance): matches human perception
gray_lum = (0.299 * R + 0.587 * G + 0.114 * B).astype(np.uint8)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(diff_rg, cmap='RdYlGn', vmin=-255, vmax=255)
axes[0, 0].set_title("R - G (Red vs Green)")

axes[0, 1].imshow(diff_rb, cmap='coolwarm', vmin=-255, vmax=255)
axes[0, 1].set_title("R - B (Red vs Blue)")

axes[1, 0].imshow(gray_avg, cmap='gray', vmin=0, vmax=255)
axes[1, 0].set_title("Grayscale (Average)")

axes[1, 1].imshow(gray_lum, cmap='gray', vmin=0, vmax=255)
axes[1, 1].set_title("Grayscale (Luminance: 0.299R + 0.587G + 0.114B)")

for ax in axes.flat:
    ax.axis("off")

plt.suptitle("Channel Arithmetic Operations", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Channel differences reveal which color dominates in each region.")
print("Luminance-weighted grayscale better matches human brightness perception.")

## 5. Channel Swapping and Manipulation

Channel swapping rearranges the R, G, B channels to create different color effects. This is useful for:
- Correcting color channel order (e.g., BGR to RGB when using OpenCV)
- Creating false-color images for visualization
- Simulating color blindness or highlighting specific features

In [ ]:
# Channel swapping examples
R_ch = img[:, :, 0]
G_ch = img[:, :, 1]
B_ch = img[:, :, 2]

# Swap R and B channels
swapped_rb = np.stack([B_ch, G_ch, R_ch], axis=2)

# Swap R and G channels
swapped_rg = np.stack([G_ch, R_ch, B_ch], axis=2)

# Isolate only the red channel (set others to zero)
red_only = np.zeros_like(img)
red_only[:, :, 0] = R_ch

# Isolate only the green channel
green_only = np.zeros_like(img)
green_only[:, :, 1] = G_ch

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

axes[0, 0].imshow(img)
axes[0, 0].set_title("Original (RGB)")

axes[0, 1].imshow(swapped_rb)
axes[0, 1].set_title("R↔B Swapped (BGR)")

axes[0, 2].imshow(swapped_rg)
axes[0, 2].set_title("R↔G Swapped (GRB)")

axes[1, 0].imshow(red_only)
axes[1, 0].set_title("Red Channel Only")

axes[1, 1].imshow(green_only)
axes[1, 1].set_title("Green Channel Only")

# Create a false-color image: invert all channels
inverted = 255 - img
axes[1, 2].imshow(inverted)
axes[1, 2].set_title("Color Inversion (complement)")

for ax in axes.flat:
    ax.axis("off")

plt.suptitle("Channel Swapping and Manipulation", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary

What we learned:
1. **Per-channel processing** = apply grayscale transformations independently to R, G, B
2. **Brightness adjustment** = add a constant to all channels (RGB) or modify intensity in HSI space (preserves hue)
3. **Contrast adjustment** = multiply channels by a scaling factor $c$
4. **Channel arithmetic** = subtract channels to find dominant colors, average channels for grayscale conversion
5. **Channel swapping** = rearrange or isolate channels for false-color visualization and color correction